# iCanClean — and why a negative control is mandatory

When you record dedicated noise-reference channels, CCA can project out the subspace they share with the scalp. The catch is that CCA will always find *some* shared subspace — so attenuation alone proves nothing.

*Deep dive behind the [five-minute demo](../meta_mne_denoise_demo.ipynb).*

## Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("demo_utils.py").exists():
    done = subprocess.run(
        ["git", "clone", "-q", "--depth", "1",
         "https://github.com/snesmaeili/mne-denoise-meta-demo.git"],
        capture_output=True, text=True)
    if done.returncode != 0:
        raise SystemExit(
            "Could not clone the demo repository. If it is still private, the Colab "
            "VM has no credentials for it -- authorising Colab lets it OPEN a "
            "notebook, not clone the repo. Make it public, or run locally."
        )
    os.chdir("mne-denoise-meta-demo")
    sys.path.insert(0, os.getcwd())
try:
    import mne_denoise  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "mne-denoise @ git+https://github.com/mne-tools/mne-denoise.git@f5b821cc2a535e84ed46085d45ea5a356dd8d548"],
                   check=True)

# The ds004505 section below reads prepared results: the raw recording is 61 GB,
# so those runs happen offline in prepare_meta_demo.py --movement.
_cache = Path(os.environ.get("MNE_DENOISE_META_DEMO_CACHE",
                             Path.home() / ".cache" / "mne-denoise" / "meta-demo"))
if not (_cache / "movement_metrics.json").exists():
    subprocess.run([sys.executable, "fetch_demo_data.py"], check=False)

import warnings, logging
import numpy as np
import matplotlib.pyplot as plt
import mne

mne.set_log_level("ERROR")
logging.getLogger("mne_denoise").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*Epochs are not baseline corrected.*")
%matplotlib inline
RANDOM_STATE = 97

## Scalp + reference, with a genuinely shared artifact

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
sfreq, n_eeg, n_ref, dur = 250.0, 32, 8, 120.0
n = int(sfreq * dur)

brain = rng.standard_normal((n_eeg, n)) * 1e-5
artifact = rng.standard_normal((4, n)) * 3e-5          # the shared source
A_eeg = rng.standard_normal((n_eeg, 4))
A_ref = rng.standard_normal((n_ref, 4))
eeg = brain + A_eeg @ artifact
ref = A_ref @ artifact + rng.standard_normal((n_ref, n)) * 5e-6

names = [f"EEG{i:03d}" for i in range(n_eeg)] + [f"N-{i:03d}" for i in range(n_ref)]
info = mne.create_info(names, sfreq, "eeg")
raw = mne.io.RawArray(np.vstack([eeg, ref]), info, verbose="ERROR")
ref_names = [c for c in raw.ch_names if c.startswith("N-")]
print(f"{n_eeg} scalp + {n_ref} reference channels, {dur:.0f} s")

## The method, and the control it must beat

The control feeds iCanClean the *same* reference channels, circularly shifted in time. Same spectra, no true alignment. Whatever it removes there, it removes by overfitting.

In [ ]:
from mne_denoise.icanclean import ICanClean

def run(data, label):
    r = mne.io.RawArray(data, info, verbose="ERROR")
    est = ICanClean(sfreq=sfreq, ref_channels=ref_names,
                    primary_channels=[c for c in raw.ch_names if c.startswith("EEG")])
    out = est.fit_transform(r)
    cleaned = out.get_data()[:n_eeg]
    kept = np.var(cleaned) / np.var(eeg)
    err = np.linalg.norm(cleaned - brain) / np.linalg.norm(brain)
    print(f"  {label:24s} removed {est.n_removed_.mean():5.2f} comp/win  "
          f"variance kept {kept:.3f}  error-vs-brain {err:.3f}")
    return est

full = np.vstack([eeg, ref])
shifted = np.vstack([eeg, np.roll(ref, int(37.0 * sfreq), axis=1)])

print(f"  {'uncorrected':24s} {'':30s} error-vs-brain "
      f"{np.linalg.norm(eeg - brain) / np.linalg.norm(brain):.3f}")
est_real = run(full, "real reference")
est_ctrl = run(shifted, "time-shifted reference")

> Here the two lines are far apart, so the attenuation is evidence. The five-minute demo shows the same separation on real blinks against real EOG electrodes: 83.1% removed with the true reference, 4.2% with one shifted by 97 s.
>
> The rest of this notebook is the case where the control comes back *negative* — and what that actually tells you.

## Canonical correlations tell you which it was

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for est, label, colour in [(est_real, "real reference", "#009E73"),
                           (est_ctrl, "time-shifted", "#D55E00")]:
    ax.plot(np.sort(est.correlations_.mean(0))[::-1], "o-", label=label, color=colour)
ax.set_xlabel("component"); ax.set_ylabel("mean squared canonical correlation")
ax.set_title("A real shared subspace separates from its own null")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.show()

## When the control comes back negative

OpenNeuro `ds004505` — table tennis with 120 scalp EEG channels, 120 dual-layer noise-reference electrodes, and independent neck EMG that no method is allowed to see. The endpoint is the coupling between each scalp channel's 30–55 Hz power and the EMG.

At package defaults, iCanClean removes 11.5% of that coupling. A reference block shifted by 100 s removes 10.5%. That is the control failing.

The raw recording is 61 GB, so these runs are prepared offline by `prepare_meta_demo.py --movement`; the numbers below are read from that cache.

In [ ]:
import json

mv = json.loads((_cache / "movement_metrics.json").read_text())
icc_rows = [r for r in mv["rows"] if r["method"].startswith("iCanClean")]

print(f"{'arm':<30s} {'attenuation':>12s} {'mean R2':>9s} {'comp/win':>9s} "
      f"{'samples/dim':>12s}")
for r in icc_rows:
    print(f"{r['method'].replace(chr(10), ' '):<30s} {r['attenuation_pct']:11.1f}% "
          f"{r['mean_r2']:9.4f} {r['components_removed']:9.2f} "
          f"{r['samples_per_dimension']:12.2f}")

d = mv["n_eeg"] + mv["n_reference"]
print(f"\n{mv['n_eeg']} primary + {mv['n_reference']} reference = {d} dimensions.")
print("At the 2 s default that is 500 samples for 240 dimensions. A CCA given")
print("barely two samples per dimension will find a 'shared' subspace in noise.")

> **What the control actually caught.** Not a broken method — a broken operating point.
>
> `stats_segment_len` computes the CCA on a wider window while still cleaning the inner one. Widen it to 30 s and the same estimator on the same data gets ~31 samples per dimension instead of 2.08. The shared subspace evaporates: mean R² falls from 0.296 to 0.026, below the 0.7 threshold, so **nothing is removed at all** — and the attenuation that looked like a result goes to zero with it.
>
> So the honest reading of `ds004505` is not "reference-based cleaning does not work here." It is: **there is no scalp↔reference subspace in this recording strong enough to survive an honest estimate**, and the 11.5% was manufactured by fitting 240 dimensions to 500 samples. Note also that the scrambled reference scores a *marginally higher* mean R² than the real one (0.2966 vs 0.2959) — with two samples per dimension, the canonical correlations are a function of the dimension counts, not of any shared source.
>
> The fix is not a bigger window for its own sake. It is to reduce the reference dimensionality to something the data can support before running the CCA at all.

## The two arms cut from the talk: DSS on the same blinks

Act 4 shows four arms. Two more were measured and cached: DSS removing the same blinks using **only the blink times**, never the EOG waveform that iCanClean and regression are handed.

That makes the linear/non-linear distinction concrete. Both arms get identical information and identical rank — the only difference is whether the criterion is a fixed covariance (`CycleAverageBias`, a closed-form generalised eigendecomposition) or one re-estimated at every iteration (`IterativeDSS` + `TanhMaskDenoiser`).

In [ ]:
E = json.loads((_cache / "eog_metrics.json").read_text())

print(f"{'arm':<38s} {'sees':<15s} {'blink removed':>13s} {'N170':>10s}")
print(f"{'uncorrected':<38s} {'--':<15s} {'--':>13s} "
      f"{E['baseline_n170_effect_uv']:+9.2f} µV")
for r in E["rows"][1:]:
    print(f"{r['method'].replace(chr(10), ' '):<38s} {r['information']:<15s} "
          f"{r['attenuation_pct']:12.1f}% {r['n170_effect_uv']:+9.2f} µV")

s = next(r["seed_spread"] for r in E["rows"] if "seed_spread" in r)
eff = np.array(s["n170_effect_uv"]); att = np.array(s["attenuation_pct"])
print(f"\nIterativeDSS is a fixed-point iteration from a random init, so it has no "
      f"single answer.\nOver {len(s['seeds'])} seeds:")
print(f"   attenuation  {att.mean():.1f}% (sd {att.std():.1f})")
print(f"   N170 effect  {eff.mean():+.3f} µV (sd {eff.std():.3f}, "
      f"range {eff.min():+.3f} to {eff.max():+.3f})")
print(f"   converged fraction {np.mean(s['converged_fraction']):.2f} at default max_iter")
print(f"\nThe reported point is seed {s['reported_seed']}, the median — picking the best "
      f"seed would be picking a result.\nLinear DSS is a closed-form eigendecomposition "
      f"and carries no spread at all.")

> **A trap worth knowing.** Do not pass `beta=beta_tanh` if you intend to reconstruct. It accelerates convergence but leaves `filters_` non-orthogonal, so `patterns_` stops being a valid inverse and `inverse_transform` returns garbage **with no error** — round-trip relative error 1.1 with it, 7e-15 without. It was caught here only by checking that the output scale matched the input.